In [5]:
!python --version

Python 3.12.13


In [6]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

In [7]:
df = pd.read_csv("Job_3_Resource_sentiment.csv")

In [8]:
df.head(5)

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [9]:
df.sample(10)

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
57124,11399,TomClancysRainbowSix,Neutral,ALRIGHT! SO great stream battle! Nothing to ex...
17484,9799,PlayStation5(PS5),Negative,"Oops, caught making people think it's a comple..."
47200,5698,HomeDepot,Positive,or
51867,10511,RedDeadRedemption(RDR),Positive,I enjoyed watching up this nice number one Rog...
35649,8121,Microsoft,Neutral,“ the Free Software Day movement is long dead....
44836,11699,Verizon,Positive,2.0 @CharityMiles for Windows. Thanks to insta...
47380,5728,HomeDepot,Positive,Today went so cool. I decided the give outdoor...
48502,5926,HomeDepot,Neutral,STOP HERE STEAL MY<unk> IN TO HOME O BYEEHEBSH...
21917,4151,CS-GO,Positive,Nice afternoon CS:GO hit thanks to the awesome...
34681,6757,Fortnite,Negative,My brother was scared.


In [10]:
df.shape

(74681, 4)

In [12]:
df.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column                                                 Non-Null Count  Dtype 
---  ------                                                 --------------  ----- 
 0   2401                                                   74681 non-null  int64 
 1   Borderlands                                            74681 non-null  object
 2   Positive                                               74681 non-null  object
 3   im getting on borderlands and i will murder you all ,  73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [15]:
df.describe()

,2401
count,74681.000000
mean,6432.640149
std,3740.423819
min,1.000000
25%,3195.000000
50%,6422.000000
75%,9601.000000
max,13200.000000


In [16]:
df.rename(columns={'Positive': 'sentiment'}, inplace=True)

In [17]:
df

,2401,Borderlands,sentiment,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...


In [18]:
df.rename(columns={'im getting on borderlands and i will murder you all ,': 'text'}, inplace=True)

In [19]:
df.sample(5)

,2401,Borderlands,sentiment,text
57573,11474,TomClancysRainbowSix,Neutral,When released you have a certain 5stack that e...
38532,5409,Hearthstone,Irrelevant,I think @ Monsanto _ HS is good for Hearthston...
53451,10784,RedDeadRedemption(RDR),Irrelevant,Cant see clearly why you would willingly get i...
35998,8179,Microsoft,Positive,Trust<unk> Technology is Important!. @satyanad...
64717,7886,MaddenNFL,Positive,You're the devil so it doesn't matter


In [20]:
df = df[['text', 'sentiment']]

In [21]:
df

,text,sentiment
0,I am coming to the borders and I will kill you...,Positive
1,im getting on borderlands and i will kill you ...,Positive
2,im coming on borderlands and i will murder you...,Positive
3,im getting on borderlands 2 and i will murder ...,Positive
4,im getting into borderlands and i can murder y...,Positive
...,...,...
74676,Just realized that the Windows partition of my...,Positive
74677,Just realized that my Mac window partition is ...,Positive
74678,Just realized the windows partition of my Mac ...,Positive
74679,Just realized between the windows partition of...,Positive


In [22]:
df.shape

(74681, 2)

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       73995 non-null  object
 1   sentiment  74681 non-null  object
dtypes: object(2)
memory usage: 1.1+ MB


In [25]:
df.describe()

,text,sentiment
count,73995,74681
unique,69490,4
top,,Negative
freq,172,22542


In [26]:
df.columns

Index(['text', 'sentiment'], dtype='object')